# Go1 joystick policy on RunPod (RTX 4090)

Trains the MuJoCo Playground `Go1JoystickFlatTerrain` policy with Brax PPO and renders rollouts.
Adapted from the official Playground locomotion Colab for a headless RunPod pod.

**Pod setup**

1. Template: any CUDA 12 image (e.g. `runpod/pytorch:2.4.0-py3.11-cuda12.4.1`), 1x RTX 4090.
2. Attach a network volume at `/workspace`. Everything outside it is wiped when the pod stops.
3. Start Jupyter on the pod and tunnel it:
   ```bash
   ssh -p <port> -L 8888:localhost:8888 root@<ip>
   jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root
   ```
4. Run **cell 1 once**, then **restart the kernel**, then run everything from cell 2 down.

Training takes roughly 7-10 minutes on a 4090 for 200M steps. Checkpoints, videos and
plots land in `/workspace/experiments/go1_joystick/<timestamp>/`.

In [ ]:
# @title 1. Install (fresh pod only, ~3 min). RESTART THE KERNEL when this finishes.
#
# Pinned to the set that is known to resolve together. jax[cuda12] pulls in the
# CUDA wheels, so the pod needs no CUDA toolkit beyond the driver.
import subprocess, sys

PKGS = [
    "jax[cuda12]==0.10.2",
    "brax==0.14.2",
    "playground==0.2.0",
    "mujoco==3.12.0",
    "mujoco-mjx==3.12.0",
    "warp-lang==1.16.0",
    "flax==0.12.8",
    "orbax-checkpoint==0.12.4",
    "ml_collections",
    "mediapy",
    "matplotlib",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PKGS], check=True)

# ffmpeg is needed by mediapy to encode the rollout videos.
if subprocess.run(["which", "ffmpeg"], capture_output=True).returncode:
    subprocess.run("apt-get update -qq && apt-get install -y -qq ffmpeg", shell=True, check=True)

print("Installed. Now restart the kernel (Kernel > Restart) and continue from cell 2.")

In [ ]:
# @title 2. Environment check: GPU, headless rendering, JAX on CUDA
import os
import subprocess

if subprocess.run(["nvidia-smi"], capture_output=True).returncode:
    raise RuntimeError("nvidia-smi failed: this pod has no visible GPU.")

# Headless rendering. Must be set BEFORE mujoco is imported anywhere in this kernel.
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

# Some container images ship the NVIDIA driver libs but not the EGL vendor ICD.
# Writing it is harmless when it already exists and we run as root on RunPod.
ICD = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(ICD):
    try:
        os.makedirs(os.path.dirname(ICD), exist_ok=True)
        with open(ICD, "w") as f:
            f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')
    except OSError as e:
        print("Could not write EGL ICD (rendering may fail):", e)

# Triton GEMM gives ~30% more steps/sec on Ada GPUs.
os.environ["XLA_FLAGS"] = os.environ.get("XLA_FLAGS", "") + " --xla_gpu_triton_gemm_any=True"

import jax

# brax 0.14.2 still calls jax.device_put_replicated, which JAX 0.10 removed from the
# public namespace. The implementation still exists; re-export it.
if not hasattr(jax, "device_put_replicated"):
    from jax._src.api import device_put_replicated as _dpr
    jax.device_put_replicated = _dpr

devices = jax.devices()
print("jax", jax.__version__, devices)
if devices[0].platform != "gpu":
    raise RuntimeError("JAX is running on CPU. Reinstall jax[cuda12] (cell 1) and restart the kernel.")

import mujoco

# Physics smoke test, then a rendering smoke test so EGL problems surface now.
m = mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom size="1"/></worldbody></mujoco>')
d = mujoco.MjData(m)
with mujoco.Renderer(m, 64, 64) as r:
    r.update_scene(d)
    assert r.render().shape == (64, 64, 3)
print("mujoco", mujoco.__version__, "- EGL rendering OK")

In [ ]:
# @title 3. Imports
import functools
import json
from datetime import datetime
from pathlib import Path

import jax
import jax.numpy as jp
import matplotlib.pyplot as plt
import mediapy as media
import mujoco
import numpy as np
from brax.io import model
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from IPython.display import clear_output, display
from mujoco_playground import registry, wrapper
from mujoco_playground._src.gait import draw_joystick_command
from mujoco_playground.config import locomotion_params

np.set_printoptions(precision=3, suppress=True, linewidth=100)

In [ ]:
# @title 4. Run directory on the persistent volume
# /workspace is the RunPod network volume. Fall back to ./experiments elsewhere.
EXPERIMENTS = Path("/workspace/experiments") if Path("/workspace").is_dir() else Path("experiments")
RUN_DIR = EXPERIMENTS / "go1_joystick" / datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Run directory:", RUN_DIR)

In [ ]:
# @title 5. Environment
env_name = "Go1JoystickFlatTerrain"
env_cfg = registry.get_default_config(env_name)

# "warp" is the fast GPU backend (default). Switch to "jax" if warp fails to compile on
# your pod; training is slower but otherwise identical.
env_cfg.impl = "warp"

env = registry.load(env_name, config=env_cfg)
print(f"obs: {env.observation_size}   act: {env.action_size}")
env_cfg

In [ ]:
# @title 6. PPO hyperparameters
SMOKE_TEST = False  # True: ~2M steps to check the pipeline end to end before a real run.

ppo_params = locomotion_params.brax_ppo_config(env_name)
if SMOKE_TEST:
    ppo_params.num_timesteps = 2_000_000
    ppo_params.num_evals = 2

with open(RUN_DIR / "env_config.json", "w") as f:
    json.dump(env_cfg.to_dict(), f, indent=2)
with open(RUN_DIR / "ppo_config.json", "w") as f:
    json.dump(ppo_params.to_dict(), f, indent=2, default=str)
ppo_params

Domain randomization (friction, link masses, torso center of mass, armature) is applied by the
Playground randomizer during training. Set `randomizer = None` below to train without it.

In [ ]:
# @title 7. Build the training function
x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]


def progress(num_steps, metrics):
    times.append(datetime.now())
    x_data.append(num_steps)
    y_data.append(float(metrics["eval/episode_reward"]))
    y_dataerr.append(float(metrics["eval/episode_reward_std"]))

    # Plain-text line survives in the notebook log even when the plot does not.
    print(f"step {num_steps:>12,d}  reward {y_data[-1]:8.3f} ± {y_dataerr[-1]:.3f}  "
          f"elapsed {times[-1] - times[0]}")

    clear_output(wait=True)
    plt.xlim([0, ppo_params["num_timesteps"] * 1.25])
    plt.xlabel("# environment steps")
    plt.ylabel("reward per episode")
    plt.title(f"y={y_data[-1]:.3f}")
    plt.errorbar(x_data, y_data, yerr=y_dataerr, color="blue")
    display(plt.gcf())


def save_checkpoint(current_step, make_policy, params):
    # Called by brax at every eval. If the pod dies mid-run, the latest one survives on /workspace.
    del make_policy
    path = RUN_DIR / f"params_{current_step:012d}"
    model.save_params(str(path), params)
    model.save_params(str(RUN_DIR / "params_latest"), params)


randomizer = registry.get_domain_randomizer(env_name)

ppo_training_params = dict(ppo_params)
network_factory = ppo_networks.make_ppo_networks
if "network_factory" in ppo_params:
    del ppo_training_params["network_factory"]
    network_factory = functools.partial(ppo_networks.make_ppo_networks, **ppo_params.network_factory)

train_fn = functools.partial(
    ppo.train,
    **ppo_training_params,
    network_factory=network_factory,
    randomization_fn=randomizer,
    progress_fn=progress,
    policy_params_fn=save_checkpoint,
)

In [ ]:
# @title 8. Train (~7-10 min on a 4090 for 200M steps; the first eval waits for JIT)
make_inference_fn, params, metrics = train_fn(
    environment=env,
    eval_env=registry.load(env_name, config=env_cfg),
    wrap_env_fn=wrapper.wrap_for_brax_training,
)
print(f"time to jit:   {times[1] - times[0]}")
print(f"time to train: {times[-1] - times[1]}")

model.save_params(str(RUN_DIR / "params_final"), params)
plt.savefig(RUN_DIR / "training_curve.png", dpi=120, bbox_inches="tight")
with open(RUN_DIR / "training_curve.json", "w") as f:
    json.dump({"steps": x_data, "reward": y_data, "reward_std": y_dataerr}, f)
print("Saved to", RUN_DIR)

### Resuming after a pod restart

If the kernel died after training, skip cell 8 and point `RESTORE_FROM` at a saved params file
(for example `/workspace/experiments/go1_joystick/<timestamp>/params_final`). The cell rebuilds
the policy network from the same hyperparameters and loads the weights. Leave it `None` to use
the policy trained above.

In [ ]:
# @title 9. (Optional) restore a trained policy instead of training
from brax.training.acme import running_statistics

RESTORE_FROM = None  # e.g. "/workspace/experiments/go1_joystick/20260906_120000/params_final"

if RESTORE_FROM is not None:
    params = model.load_params(RESTORE_FROM)
    obs_shape = {k: (v,) if isinstance(v, int) else tuple(v) for k, v in env.observation_size.items()}
    ppo_network = network_factory(
        obs_shape, env.action_size, preprocess_observations_fn=running_statistics.normalize
    )
    make_inference_fn = ppo_networks.make_inference_fn(ppo_network)
    print("Restored policy from", RESTORE_FROM)

## Rollout and render

The eval environment gets random pushes on the torso (perturbations) and a wider yaw command
range so the video shows how the policy recovers. Videos are written to the run directory and
shown inline.

In [ ]:
# @title 10. Eval environment with perturbations
eval_cfg = registry.get_default_config(env_name)
eval_cfg.impl = env_cfg.impl
eval_cfg.pert_config.enable = True
eval_cfg.pert_config.velocity_kick = [3.0, 6.0]
eval_cfg.pert_config.kick_wait_times = [5.0, 15.0]
eval_cfg.command_config.a = [1.5, 0.8, 2 * jp.pi]
eval_env = registry.load(env_name, config=eval_cfg)

velocity_kick_range = [0.0, 0.0]  # [0, 0] disables the kicks; try [3.0, 6.0] for pushes.
kick_duration_range = [0.05, 0.2]

jit_reset = jax.jit(eval_env.reset)
jit_step = jax.jit(eval_env.step)
jit_inference_fn = jax.jit(make_inference_fn(params, deterministic=True))


def sample_pert(rng, state):
    rng, key1, key2 = jax.random.split(rng, 3)
    pert_mag = jax.random.uniform(key1, minval=velocity_kick_range[0], maxval=velocity_kick_range[1])
    duration_seconds = jax.random.uniform(key2, minval=kick_duration_range[0], maxval=kick_duration_range[1])
    state.info["pert_mag"] = pert_mag
    state.info["pert_duration"] = jp.round(duration_seconds / eval_env.dt).astype(jp.int32)
    state.info["pert_duration_seconds"] = duration_seconds
    return rng


def torso_marker(state, cmd):
    # Draws the joystick command arrow above the torso in the rendered video.
    xyz = np.array(state.data.xpos[eval_env._torso_body_id]) + np.array([0, 0, 0.2])
    x_axis = state.data.xmat[eval_env._torso_body_id, 0]
    yaw = -np.arctan2(x_axis[1], x_axis[0])
    return functools.partial(
        draw_joystick_command, cmd=cmd, xyz=xyz, theta=yaw,
        scl=abs(float(cmd[0])) / eval_cfg.command_config.a[0],
    )


def render(rollout, modify_scene_fns, filename, render_every=2):
    fps = 1.0 / eval_env.dt / render_every
    scene_option = mujoco.MjvOption()
    scene_option.geomgroup[2] = True
    scene_option.geomgroup[3] = False
    scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
    scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False
    scene_option.flags[mujoco.mjtVisFlag.mjVIS_PERTFORCE] = True
    frames = eval_env.render(
        rollout[::render_every], camera="track", scene_option=scene_option,
        width=640, height=480, modify_scene_fns=modify_scene_fns[::render_every],
    )
    media.write_video(RUN_DIR / filename, frames, fps=fps)
    print("Wrote", RUN_DIR / filename)
    media.show_video(frames, fps=fps, loop=False)

In [ ]:
# @title 11. Constant command rollout
x_vel = 0.0    # m/s forward
y_vel = 0.0    # m/s lateral
yaw_vel = 3.14 # rad/s

command = jp.array([x_vel, y_vel, yaw_vel])
rng = jax.random.PRNGKey(0)
rollout, modify_scene_fns = [], []
swing_peak, linvel, angvel, rewards = [], [], [], []

state = jit_reset(rng)
if state.info["steps_since_last_pert"] < state.info["steps_until_next_pert"]:
    rng = sample_pert(rng, state)
state.info["command"] = command

for i in range(eval_cfg.episode_length):
    if state.info["steps_since_last_pert"] < state.info["steps_until_next_pert"]:
        rng = sample_pert(rng, state)
    act_rng, rng = jax.random.split(rng)
    ctrl, _ = jit_inference_fn(state.obs, act_rng)
    state = jit_step(state, ctrl)
    state.info["command"] = command

    rollout.append(state)
    modify_scene_fns.append(torso_marker(state, state.info["command"]))
    swing_peak.append(state.info["swing_peak"])
    linvel.append(eval_env.get_global_linvel(state.data))
    angvel.append(eval_env.get_gyro(state.data))
    rewards.append({k[7:]: float(v) for k, v in state.metrics.items() if k.startswith("reward/")})

print("mean reward terms over the episode:")
for k in rewards[0]:
    print(f"  {k:18s} {np.mean([r[k] for r in rewards]):8.4f}")

render(rollout, modify_scene_fns, "rollout_constant_cmd.mp4")

In [ ]:
# @title 12. Foot heights and velocity tracking for the rollout above
def plot_tracking(swing_peak, linvel, angvel, command, filename):
    swing_peak = jp.array(swing_peak)
    names = ["FR", "FL", "RR", "RL"]
    colors = ["r", "g", "b", "y"]
    fig, axs = plt.subplots(2, 2)
    for i, ax in enumerate(axs.flat):
        ax.plot(swing_peak[:, i], color=colors[i])
        ax.set_ylim([0, eval_cfg.reward_config.max_foot_height * 1.25])
        ax.axhline(eval_cfg.reward_config.max_foot_height, color="k", linestyle="--")
        ax.set_title(names[i])
        ax.set_xlabel("time")
        ax.set_ylabel("height")
    plt.tight_layout()
    fig.savefig(RUN_DIR / f"{filename}_feet.png", dpi=120)
    plt.show()

    smooth = lambda x: jp.convolve(x, jp.ones(10) / 10, mode="same")
    linvel_x = smooth(jp.array(linvel)[:, 0])
    linvel_y = smooth(jp.array(linvel)[:, 1])
    angvel_yaw = smooth(jp.array(angvel)[:, 2])

    fig, axes = plt.subplots(3, 1, figsize=(10, 10))
    for ax, series, lim, label in zip(
        axes, [linvel_x, linvel_y, angvel_yaw], eval_cfg.command_config.a, ["dx", "dy", "dyaw"]
    ):
        ax.plot(series)
        ax.set_ylim(-lim, lim)
        ax.set_ylabel(label)
    for i, ax in enumerate(axes):
        ax.axhline(float(command[i]), color="red", linestyle="--")
    fig.savefig(RUN_DIR / f"{filename}_tracking.png", dpi=120)
    plt.show()


plot_tracking(swing_peak, linvel, angvel, command, "rollout_constant_cmd")

In [ ]:
# @title 13. Ramp the forward velocity command
rng = jax.random.PRNGKey(0)
rollout, modify_scene_fns = [], []
swing_peak, linvel, angvel = [], [], []

x = -0.25
command = jp.array([x, 0, 0])
state = jit_reset(rng)
for i in range(1_400):
    # +0.25 m/s every 200 steps (4 s).
    if i % 200 == 0:
        x += 0.25
        print(f"Setting x to {x}")
        command = jp.array([x, 0, 0])
    state.info["command"] = command
    if state.info["steps_since_last_pert"] < state.info["steps_until_next_pert"]:
        rng = sample_pert(rng, state)
    act_rng, rng = jax.random.split(rng)
    ctrl, _ = jit_inference_fn(state.obs, act_rng)
    state = jit_step(state, ctrl)

    rollout.append(state)
    modify_scene_fns.append(torso_marker(state, command))
    swing_peak.append(state.info["swing_peak"])
    linvel.append(eval_env.get_global_linvel(state.data))
    angvel.append(eval_env.get_gyro(state.data))

plot_tracking(swing_peak, linvel, angvel, command, "rollout_ramp")
render(rollout, modify_scene_fns, "rollout_ramp.mp4")

Done. Everything from this run is in `RUN_DIR` on the network volume:
`params_final`, `params_latest`, the per-eval `params_*` checkpoints, both configs,
the training curve, videos and plots. **Stop the pod when you are finished.**